In [38]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sys

# dev = "sincos_laptop"
# title = f"UR10e IK benchmark - sin,cos return - Laptop"

# dev = "sincos_cluster"
# title = f"UR10e IK benchmark - sin,cos return - Cluster"

dev = "cluster"
title = f"UR10e IK benchmark - Cluster"

fname = f"benchmark_results_{dev}.json"

with open(fname) as f:
    raw = json.load(f)

data = {}
for solver, ops in raw.items():
    data[solver] = {}
    for op, vals in ops.items():
        data[solver][op] = {int(k): v for k, v in vals.items()}

SOLVERS = [
    ("jax_cpu",       "JAX CPU",      "#378ADD", "-",  "o"),
    ("jax_gpu",       "JAX GPU",      "#1D9E75", "-",  "s"),
    ("numba_serial",  "Numba serial CPU", "#BA7517", "-",  "^"),
    ("numba_prange",  "Numba prange CPU", "#D85A30", "--", "D"),
]
SCALE_SOLVERS = {"jax_cpu", "jax_gpu", "numba_serial"}

for op in ("fk", "ik"):
    fig, ax = plt.subplots(figsize=(7, 5))
    lines, labels, scale_vals = [], [], []

    for solver_key, label, color, ls, marker in SOLVERS:
        if solver_key not in data:
            continue
        op_data = data[solver_key].get(op, {})
        if not op_data:
            continue
        Bs   = sorted(op_data)
        mus  = [op_data[B] for B in Bs]
        mcps = [1 / v for v in mus]   # M calls/s  (1 / µs = 1e6 calls/s = 1 Mcall/s)
        print(op, mcps)
        l, = ax.plot(Bs, mcps, color=color, linestyle=ls,
                     marker=marker, markersize=5, linewidth=1.8, label=label)
        lines.append(l); labels.append(label)
        if solver_key in SCALE_SOLVERS:
            scale_vals.extend(mcps)

    # ── left axis : M calls/s ─────────────────────────────────────────────────
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel("Throughput (M calls/s)", fontsize=11)
    ax.set_title(title, fontsize=12, fontweight="500")
    ax.grid(True, which="both", linestyle=":", linewidth=0.5, alpha=0.6)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:g}"))
    ax.yaxis.set_minor_formatter(ticker.NullFormatter())

    if scale_vals:
        ax.set_ylim(min(scale_vals) * 0.5, max(scale_vals) * 2.0)

    ax.legend(lines, labels, fontsize=9, loc="upper left",
              framealpha=0.85, edgecolor="#cccccc")
    plt.tight_layout()
    for ext in ("png", "pdf"):
        out = f"benchmark_{dev}_{op}.{ext}"
        plt.savefig(out, bbox_inches="tight", dpi=150)
        print(f"Saved {out}")
    plt.close()

fk [0.09932397428426067, 0.38630754596150385, 1.5384670450775173, 3.1971097728927504, 5.482563328177434, 6.273933936237078, 6.452076288766237, 11.13897350994401, 24.48790407149607, 26.86905654147327, 16.675187889944432, 16.514504613102503]
fk [0.007678901981958466, 0.028008068576272846, 0.11522071295203348, 0.43017122463738067, 1.8450054540368612, 6.84105602996469, 27.33490741146354, 79.77601119362778, 163.54108483026297, 214.22910373541913, 133.1483388651077, 747.3404281871367]
fk [0.1564480128219138, 0.4167544619265944, 0.731860195064805, 1.087567956477959, 1.0950983812993635, 1.1026016689532414, 1.0077837492874568, 1.084613353333979]
fk [0.00016667563938203188, 0.10520112172927958, 0.21450001400253768, 0.3411667835762589, 0.39708483505028425, 0.4039572615444515, 0.4125978943802308, 0.41515993116849986]
Saved benchmark_cluster_fk.png
Saved benchmark_cluster_fk.pdf
ik [0.0436615749393867, 0.16348856793201563, 0.5115092446472368, 0.894217234971291, 0.9207872775895141, 1.134244754200310

In [40]:
s = [0.008553113438646627, 0.031406577852526894, 0.13346399529998068, 0.4719136562839607, 1.9210124174816396, 6.421954311014674, 17.48823596981144, 81.86267274433192, 192.96296235835825, 379.70153239464884, 451.9055021496766, 473.0831704564279]
b = 4**np.array(range(len(s)))
for x,y in zip(b,s):
    print(x, round(y,2))

1 0.01
4 0.03
16 0.13
64 0.47
256 1.92
1024 6.42
4096 17.49
16384 81.86
65536 192.96
262144 379.7
1048576 451.91
4194304 473.08
